# Booking.com Flights – Datenbereinigung & erste EDA (FHNW LO4)

**Mini-Challenge:** Vergleich von Direktflug-Angeboten auf Booking.com Flights.

**Was ich mache:** Ich vergleiche **Nur-Direktflüge** mit Abflug **Zürich (ZRH)** versus **Basel (BSL)** Richtung mehrerer **europäischer Destinationen**.

**Website:** [Booking.com Flights](https://www.booking.com/flights)

**Technik:** **Selenium** öffnet die Seite und liest die Suchresultate aus; die Suche selbst war bei mir absichtlich teilweise **manuell** (halbautomatisch), weil die Seite dynamische React-Komponenten hat und eine gleichzeitige Suche mit mehreren Abflughäfen in Tests instabil war.

**Zeitraum der Abflugdaten:** 13.06.2026 bis 24.07.2026

In diesem Notebook arbeite ich zuerst mit den Rohdaten (`raw`), lade dann die bereinigte Datei (`processed`) und mache ein paar erste Plots.

## 2. Datenerhebung (Data Collection)

So ist mein **halbautomatischer** Ablauf gelaufen:

- **Selenium** öffnet Booking.com Flights, akzeptiert Cookies (falls ein Banner kommt) und stellt **«Nur Hinflug»** sowie **«Nur Direktflüge»** ein.
- **Abflug­ort, Ziele und Datum** habe ich im Browser **von Hand** eingegeben und die Suche **manuell** gestartet – so bleibt der Ablauf stabiler als bei der vollen Automatisierung.
- **Trefferliste & Pagination:** Sobald Resultate da waren, hat das Skript die Karten **automatisch** ausgelesen und über **«Weiter»** weitere Seiten geholt (bis zu einem Limit).
- Ich habe **keine Captchas umgangen** und keine Tricks an Sicherheitsmechanismen versucht; falls die Seite blockiert, habe ich gestoppt.
- Die CSV wurde **pro Durchlauf angehängt** (`append`) – mehrere Sessions können also in **einer** Raw-Datei landen.

Damit entspricht mein Datensatz eher einer **Momentaufnahme** mehrerer Suchläufe als einer repräsentativen Marktstudie – aber für einen Einstieg und für LO4 reicht mir das.


## 3. Rohdaten laden (Load Raw Data)

Ich lese die Datei `data/raw/booking_flights_raw.csv`. Das ist exakt das, was der Scraper geschrieben hat (plus ggf. meine manuellen Metadaten wie `origin` und `departure_date` pro Durchlauf).

**Hinweis:** Wenn ich Jupyter vom Ordner `notebooks/` starte, zeigt der Code unten automatisch auf das Projektverzeichnis mit dem Ordner `data/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Projektroot finden (Notebook in notebooks/ oder Start im Root)
_root = Path.cwd()
if not (_root / "data").is_dir():
    _root = _root.parent

RAW_CSV = _root / "data" / "raw" / "booking_flights_raw.csv"
PROC_CSV = _root / "data" / "processed" / "booking_flights_processed.csv"

print("Projektroot:", _root.resolve())

df_raw = pd.read_csv(RAW_CSV, encoding="utf-8")

display(df_raw.head())
df_raw.info()
print("Zeilen × Spalten:", df_raw.shape)

**Kurz zur Raw-Struktur:** Die wichtigsten Spalten sind Metadaten zum Durchlauf (`scrape_timestamp`, `origin`, `destination_city`, `departure_date`, …) plus die aus der Trefferliste gelesenen Felder (`price_text`, `carrier`, Zeiten, Flughäfen, `duration`, `stops`). `price_text` ist noch Text (z. B. «87 CHF»), und `destination_city` kann eine **komplette Suchliste** sein – für die Auswertung nutze ich später lieber `destination_group` aus dem **Zielflughafen** (`destination_airport`).

## 4. Datenbereinigung / Processing

Die bereinigte Datei entsteht mit `python -m flight_scraper process` und liegt unter `data/processed/booking_flights_processed.csv`. Unten lade ich sie und liste **zusätzliche** Spalten auf, die beim Processing erzeugt wurden.

In [ ]:
from IPython.display import display

df_proc = pd.read_csv(PROC_CSV, encoding="utf-8")

new_cols = [
    "departure_date_clean",
    "weekday",
    "weekday_num",
    "booking_lead_days",
    "price",
    "currency",
    "duration_minutes",
    "destination_group",
]
present = [c for c in new_cols if c in df_proc.columns]
missing = [c for c in new_cols if c not in df_proc.columns]

print("Neue / erwartete Spalten vorhanden:", present)
if missing:
    print("Fehlt (evtl. alte Processed-Datei?):", missing)

display(df_proc.head())
print("Shape processed:", df_proc.shape)

**Was passiert ist (in eigenen Worten):**

- **`price_text` → `price` + `currency`:** Aus Strings wie `87 CHF` werden eine Zahl und der Währungscode.
- **`duration` → `duration_minutes`:** Deutsche Texte («1 Std. 55 Min.») werden in Minuten umgerechnet.
- **`departure_date`:** wird normalisiert; es gibt **`departure_date_clean`** im Format `YYYY-MM-DD`.
- **Bekannter Tippfehler:** `"2026-24"` wird vor dem Parsen auf **`2026-07-24`** gemappt (kleine Korrekturtabelle im Code).
- **`weekday` / `weekday_num`:** Wochentag aus dem Abflugdatum (Montag = 0, …).
- **`booking_lead_days`:** Tage zwischen Scraping-Datum und Abflugdatum (nur sinnvoll, wenn beide Werte da sind).
- **`destination_group`:** grobe Stadtgruppe aus **`destination_airport`** (z. B. LGW → London), weil in einer Suche mehrere Städte gleichzeitig vorkommen können.
- **Duplikate:** identische Kombinationen aus Kernfeldern werden **einmal** behalten.

Wenn das für meine Note stimmt, habe ich eine einheitliche Basis für die Plots.

## 5. Datenqualität (Data Quality Checks)

Ich vergleiche Roh- und bereinigte Zeilenanzahl, schaue auf fehlende Werte und auf Verteilungen nach `origin`, Datum und `destination_group`. So erkenne ich schnell, ob etwas «off» wirkt — z. B. viele fehlende Preise oder zu wenige BSL-Zeilen.

In [ ]:
from IPython.display import display

print("Zeilen raw:     ", len(df_raw))
print("Zeilen processed:", len(df_proc))

miss = df_proc.isna().sum()
miss_nonzero = miss[miss > 0].sort_values(ascending=False)
print("\nFehlende Werte (nur Spalten mit Lücken):")
print(miss_nonzero)

print("\n--- Zeilen pro origin (processed) ---")
print(df_proc["origin"].value_counts())

dc = pd.to_datetime(df_proc["departure_date_clean"], errors="coerce")
print("\n--- Datumsbereich Abflug (departure_date_clean) ---")
print("Min:", dc.min(), "| Max:", dc.max())

tbl_day_origin = (
    df_proc.groupby(["departure_date_clean", "origin"])
    .size()
    .reset_index(name="n_flights")
    .sort_values(["departure_date_clean", "origin"])
)
print("\n--- Flüge pro Abflugtag und origin (Auszug) ---")
display(tbl_day_origin.head(15))

print("\n--- Flüge pro destination_group ---")
print(df_proc["destination_group"].value_counts())

**Interpretation:** Wenn die processed-Datei etwas weniger Zeilen hat als raw, liegt das meist an **ungültigen Datums­werten** oder **Duplikaten** – das ist gewollt. Für meine Forschungsfrage ist wichtig: **ZRH hat typischerweise mehr Treffer als BSL.** Das ist in meinen Daten **kein Sampling-Fehler**, sondern ein **echter Unterschied im Suchergebnis** (mehr angezeigte Direktflüge pro Suchlauf). Ich gleiche die Stichprobe **nicht künstlich** an — ich interpretiere also «mehr Zeilen = mehr Angebote in meinem Snapshot», nicht «gleiche Grundgesamtheit».

Wenn `price` oder `booking_lead_days` Lücken haben, sollte ich in späteren Auswertungen vorsichtig sein oder Zeilen filtern.

## 6. EDA 1 — Anzahl Flüge pro Tag nach Flughafen

Ich zähle pro **Abflugdatum** und **Origin**, wie viele Zeilen (angezeigte Flüge) in meinem Datensatz vorkommen. Das ist **kein** exakter Marktanteil, aber ein guter erster Eindruck: Wo kommen mehr Karten in der Suchliste vor?

In [ ]:
counts = (
    df_proc.groupby(["departure_date_clean", "origin"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

dates = pd.to_datetime(counts.index)

fig, ax = plt.subplots(figsize=(11, 4))
for origin in counts.columns:
    ax.plot(dates, counts[origin], marker="o", markersize=4, label=origin)

ax.set_title("Anzahl Flüge pro Abflugtag nach Origin (processed)")
ax.set_xlabel("Abflugdatum")
ax.set_ylabel("Anzahl Treffer in den Daten")
ax.legend(title="Origin")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

**Was ich daraus lese:** Meist ist zu sehen, dass **ZRH über die Tage mehr Einträge** hat als **BSL** — das passt zu meiner Erwartung aus der Datenerhebung (verschieden viele Resultate je Suche). Manchmal gibt es **Lücken oder Zacken** an einzelnen Tagen: das kann daran liegen, **welche** Tage ich überhaupt gesucht habe, oder daran, dass Booking nicht jeden Tag identisch sortiert. Für die Mini-Challenge reicht mir: **Ich vergleiche die Flughäfen ehrlich mit ungleicher Stichprobengrösse** und erwähne das auch im Text.

## 7. EDA 2 — Preisvergleich ZRH vs. BSL

Ich schaue mir **Lage** und **Streuung** der Preise an. Der **Median** ist oft robuster als der Durchschnitt («ein teurer Ausreisser zieht den Mean hoch»). Die **Box** zeigt Quartile; der **Whisker** die Spanne der typischen Werte (matplotlib-Standard).

In [ ]:
price_stats = (
    df_proc.dropna(subset=["price", "origin"])
    .groupby("origin")["price"]
    .agg(count="count", mean="mean", median="median", min="min", max="max", std="std")
    .round(2)
)
display(price_stats)

origins = sorted(df_proc["origin"].dropna().unique())
series_list = [
    df_proc.loc[df_proc["origin"] == o, "price"].dropna().values for o in origins
]

fig, ax = plt.subplots(figsize=(6, 5))
bp = ax.boxplot(series_list, labels=origins, patch_artist=True)
for box in bp["boxes"]:
    box.set_facecolor("#cfe2f3")

ax.set_title("Preisverteilung nach Origin")
ax.set_ylabel("Preis (wie in Rohdaten, meist CHF)")
plt.tight_layout()
plt.show()

**Interpretation (vorsichtig):** Wenn eine Box **tiefer** liegt, sind die angezeigten Preise in meinem Sample oft **eher günstiger** — aber: **andere Stichprobengrösse** (mehr ZRH-Flüge) kann die Form der Verteilung beeinflussen. Ich **normiere nicht** auf «gleich viele Flüge», weil das hier wie Künsteln wirken würde. Besser: **Median/Vergleich transparent erklären** und im Limitations-Teil die ungleiche Trefferzahl erwähnen.

## 8. EDA 3 — Preise nach Destination (`destination_group`)

Jetzt splitte ich nach **Stadtgruppe** (aus dem Zielflughafen abgeleitet) **und** nach **Origin**. So sehe ich z. B., ob London in meinem Sample teurer ist als Berlin — immer **innerhalb desselben Scraping-Setups**.

In [ ]:
sub = df_proc.dropna(subset=["price", "destination_group", "origin"])
tbl_dest = (
    sub.groupby(["destination_group", "origin"])["price"]
    .agg(mean_price="mean", median_price="median", n="count")
    .round(2)
)
display(tbl_dest)

dest_order = sorted(sub["destination_group"].unique())
origins = sorted(sub["origin"].unique())

x = np.arange(len(dest_order))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5))
for i, origin in enumerate(origins):
    means = (
        sub.loc[sub["origin"] == origin]
        .groupby("destination_group")["price"]
        .mean()
        .reindex(dest_order)
    )
    offset = (i - (len(origins) - 1) / 2) * width
    ax.bar(x + offset, means, width, label=origin)

ax.set_xticks(x)
ax.set_xticklabels(dest_order, rotation=25, ha="right")
ax.set_ylabel("Durchschnittspreis")
ax.set_title("Ø Preis nach destination_group und origin")
ax.legend(title="Origin")
plt.tight_layout()
plt.show()

**Lesart:** Gruppen mit **höherem Mittel** sind in meinem Snapshot eher **teuer** — das muss aber nicht heissen, dass die Stadt «generell teurer» ist; Timing, Airline-Mix und Suchfilter spielen mit. Wenn ich im Report eine Destination als «günstig» beschreibe, erwähne ich am besten **immer**, dass es Booking.com-Daten mit Direktfiltern sind.

## 9. EDA 4 — Airlines (Carrier)

Welche **Airlines** tauchen am häufigsten auf? Gibt es Unterschiede zwischen **ZRH** und **BSL**? Das ist deskriptiv — ich behaupte hier keine Marktanteile.

In [ ]:
top_n = 10

fig, ax = plt.subplots(figsize=(8, 4))
df_proc["carrier"].value_counts().head(top_n).sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"Top {top_n} Airlines (alle Origins)")
ax.set_xlabel("Anzahl Flüge in den Daten")
plt.tight_layout()
plt.show()

for origin in sorted(df_proc["origin"].dropna().unique()):
    print(f"\n--- Top 5 Carrier für {origin} ---")
    print(df_proc.loc[df_proc["origin"] == origin, "carrier"].value_counts().head(5))

carrier_price = (
    df_proc.dropna(subset=["carrier", "price"])
    .groupby("carrier", observed=True)
    .agg(n=("price", "count"), mean_price=("price", "mean"))
)
carrier_price = carrier_price[carrier_price["n"] >= 8].sort_values("mean_price")
print("\nCarrier mit ≥8 Flügen — günstigste / teuerste Ø-Preise (Auszug):")
display(carrier_price.head(5))
display(carrier_price.tail(5))

**Interpretation:** Oft dominieren **Low-Cost** und **Network-Carrier** je nach Route — das Pattern kann für **ZRH** anders aussehen als für **BSL**, weil Angebot und Hub-Logik anders sind. Wenn z. B. eine Airline bei BSL nur selten vorkommt, ist der **Durchschnittspreis** dort schnell «verzerrt» (wenige Flüge) — deshalb schaue ich für Preise pro Carrier nur bei **mindestens 8** Flügen (die Schwelle kann ich im Code anpassen).

## 10. EDA 5 — Wochentag & Buchungsvorlauf

Hier wird es «ein bisschen Data Science»: **Wochentag** und **wie weit im Voraus** gebucht wurde (`booking_lead_days`) könnten mit dem Preis zusammenhängen — **ohne** dass ich kausal behaupte, *dass* der Wochentag *den* Preis **verursacht** (Stichwort: Störvariablen wie Destination und Airline).

In [ ]:
wd_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

sub_wd = df_proc.dropna(subset=["price", "weekday"])
mean_by_wd = sub_wd.groupby("weekday", observed=True)["price"].mean().reindex(wd_order)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(mean_by_wd)), mean_by_wd.values)
ax.set_xticks(range(len(mean_by_wd)))
ax.set_xticklabels(mean_by_wd.index, rotation=35, ha="right")
ax.set_title("Durchschnittspreis nach Wochentag (Abflugdatum)")
ax.set_xlabel("weekday")
ax.set_ylabel("Ø Preis")
plt.tight_layout()
plt.show()

sub_sc = df_proc.dropna(subset=["price", "booking_lead_days"]).copy()
sub_sc["booking_lead_days"] = pd.to_numeric(sub_sc["booking_lead_days"], errors="coerce")
sub_sc = sub_sc.dropna(subset=["booking_lead_days"])

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.scatter(
    sub_sc["booking_lead_days"],
    sub_sc["price"],
    alpha=0.25,
    s=12,
)
ax2.set_title("Preis vs. Buchungsvorlauf (booking_lead_days)")
ax2.set_xlabel("Tage zwischen Scrape-Datum und Abflug")
ax2.set_ylabel("Preis")
plt.tight_layout()
plt.show()

**Vorsicht beim Storytelling:** Wenn der Scatter «von links oben nach rechts unten» aussieht, **kann** das auf *früher buchen ⇒ günstiger* hindeuten — aber genauso gut stecken **andere Muster** dahinter (z. B. bestimmte Destinationen werden früher gebucht). Für LO4 reicht mir: **Ich beschreibe**, was ich sehe, und schreibe dazu, dass **Korrelation ≠ Kausalität**.

## 11. Limitationen (was die Daten *nicht* können)

- **Anbieter:** Alles stammt von **Booking.com Flights**. Andere Buchungsportale oder Airline-Direktpreise sind nicht drin.
- **Preise:** Das sind **Momentaufnahmen** zum **Scrape-Zeitpunkt**, nicht «die Wahrheit für den ganzen Markt».
- **Erhebung:** **Halbautomatisch** — also von mir menschlich beeinflusst (wann gesucht wurde, welche UI-Kombination, wie viele Pagination-Klicks, …).
- **Ungleiche Treffer:** **ZRH** und **BSL** haben oft **unterschiedlich viele** Resultate; ich behandle das als **Merkmal meiner Realität** im Sample, nicht als «Fehler».
- **Destination:** Oft habe ich **mehrere Ziele in einer Suche** kombiniert; für die Analyse habe ich **`destination_group`** aus dem **Zielflughafen** (`destination_airport`) gebaut — das ist pragmatisch, aber nicht perfekt.
- **Technik:** Booking kann **`data-testid` / Selektoren** ändern — der Scraper kann dann brechen oder Felder falsch lesen.
- **Ethik / ToS:** Ich wollte **keine Sicherheitsmechanismen umgehen**; wenn die Seite blockt, ist Stopp angesagt.

Kurz: Das ist für mich eine **saubere erste empirische Übung**, keine vollständige Wettbewerbsanalyse.

## 12. Fazit

**Zurück zur Frage:** Wie unterscheiden sich **Direktflug-Angebote** ab **Zürich (ZRH)** vs. **Basel (BSL)** in meinem Daten-Snapshot?

- **Angebot / Trefferzahlen:** Meist sehe ich **mehr Karten** für **ZRH** — in meinem Setup spiegelt das wider, was Booking in meinen Suchläufen gezeigt hat (nicht zwingend «die ganze Welt»).
- **Preise:** Boxplot und Kennzahlen (**Median/Spread**) geben einen **Eindruck**, ob ein Flughafen eher günstigere Direktpreise anzeigt — aber ohne gleiche Stichprobeneigenschaften bleibt das **deskriptiv**, nicht «final bewiesen».
- **Destination & Airlines:** Oft gibt es klare **Unterschiede zwischen Destinationen** und im **Carrier-Mix** — das hilft zu erklären, warum ein Flughafen «im Schnitt» anders aussehen kann.

Für meinen Report lohnt sich ein roter Faden: **Datenquelle → Cleaning → erste Plots → Limitationen → vorsichtige Schlussfolgerung**. Viel Erfolg bei LO4!